In [66]:
import numpy as np 
import pandas as pd
from pathlib import Path

In [67]:
df = pd.read_csv("../data/dirty_cafe_sales.csv")  # Loading DataSet
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [68]:
print("Rows : ", df.shape[0])
print("Columns : ", df.shape[1])

print("\nColumn Names : ")
print(df.columns.tolist())

print("\nData Types : ")
print(df.dtypes)

Rows :  10000
Columns :  8

Column Names : 
['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Data Types : 
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object


In [69]:
df.info()  # Data Information

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [70]:
df.describe(include="all")

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [71]:
df.shape

(10000, 8)

In [72]:
missing_value = df.isnull().sum()  # Checking missing Values
missing_value

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [73]:
df.dtypes

Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object

In [74]:
duplicates = df.duplicated().sum()
print("Duplicate rows : ", duplicates)

Duplicate rows :  0


In [75]:
for column in df.columns:
    print(f"\n{column}:")
    print(df[column].unique()[:20])
# Unique Values


Transaction ID:
['TXN_1961373' 'TXN_4977031' 'TXN_4271903' 'TXN_7034554' 'TXN_3160411'
 'TXN_2602893' 'TXN_4433211' 'TXN_6699534' 'TXN_4717867' 'TXN_2064365'
 'TXN_2548360' 'TXN_3051279' 'TXN_7619095' 'TXN_9437049' 'TXN_8915701'
 'TXN_2847255' 'TXN_3765707' 'TXN_6769710' 'TXN_8876618' 'TXN_3709394']

Item:
['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']

Quantity:
['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]

Price Per Unit:
['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']

Total Spent:
['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']

Payment Method:
['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]

Location:
['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']

Transaction Date:
['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11'
 '2023-03-31' '2023-10-06' '2023-10-28' '2023-07-28' '2023-12-31'
 '2023-11-07' 'ERROR' '2023-05-

In [76]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

df.columns.tolist()


['transaction_id',
 'item',
 'quantity',
 'price_per_unit',
 'total_spent',
 'payment_method',
 'location',
 'transaction_date']

In [77]:
df = df.drop_duplicates()
print("Rows after removing Duplicates : ", len(df))

Rows after removing Duplicates :  10000


In [78]:
text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

In [79]:
invalid_values = [
    "",
    "nan",
    "none",
    "n/a",
    "na",
    "unknown",
    "error"
]
for column in df.columns:
    if df[column].dtype == "string":
        df[column] = df[column].replace(invalid_values, pd.NA)

In [80]:
numeric_columns = [
    "quantity",
    "price_per_unit",
    "total_spent"
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

In [81]:
if "transaction_date" in df.columns:
    df["transaction_date"] = pd.to_datetime(
        df["transaction_date"],
        errors="coerce"
    )

elif "date" in df.columns:
    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

In [82]:
df.isnull().sum()

transaction_id         0
item                 333
quantity             479
price_per_unit       533
total_spent          502
payment_method      2579
location            3265
transaction_date     460
dtype: int64

In [83]:
required_columns = [
    "quantity",
    "price_per_unit"
]

existing_required_columns = [
    column
    for column in required_columns
    if column in df.columns
]

df = df.dropna(
    subset=existing_required_columns
)

if "quantity" in df.columns:
    df = df[df["quantity"] > 0]

if "price_per_unit" in df.columns:
    df = df[df["price_per_unit"] > 0]

print("Rows after cleaning:", len(df))

Rows after cleaning: 9006


In [84]:
if "quantity" in df.columns and "price_per_unit" in df.columns:
    df["calculated_revenue"] = (
        df["quantity"] * df["price_per_unit"]
    )

df.head()   # Calculating Revenue

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,calculated_revenue
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,4.0
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,12.0
2,TXN_4271903,Cookie,4,1.0,<NA>,Credit Card,In-store,2023-07-19,4.0
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27,10.0
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,4.0


In [85]:
if "total_spent" in df.columns and "calculated_revenue" in df.columns:
    df["revenue_difference"] = (
        df["total_spent"] - df["calculated_revenue"]
    )

    df[
        [
            "total_spent",
            "calculated_revenue",
            "revenue_difference"
        ]
    ].head(10)
# Compare With Existing Total

In [86]:
if "calculated_revenue" in df.columns:
    df["revenue"] = df["calculated_revenue"]

df.head()
# Final Revenue Column

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,calculated_revenue,revenue_difference,revenue
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,4.0,0.0,4.0
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,12.0,0.0,12.0
2,TXN_4271903,Cookie,4,1.0,<NA>,Credit Card,In-store,2023-07-19,4.0,<NA>,4.0
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27,10.0,0.0,10.0
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,4.0,0.0,4.0


In [87]:
total_revenue = df["revenue"].sum()
total_transactions = len(df)
average_transaction = df["revenue"].mean()

print("Total Revenue:", total_revenue)
print("Total Transactions:", total_transactions)
print("Average Transaction Value:", average_transaction) 
# Overall Sales Summary 

Total Revenue: 80478.5
Total Transactions: 9006
Average Transaction Value: 8.936098156784366


In [88]:
if "item" in df.columns:
    top_items = (
        df.groupby("item")["quantity"]
        .sum()
        .sort_values(ascending=False)
    )

    print(top_items.head(10))
# Top-Selling Items

item
Coffee      3212
Juice       3187
Cake        3180
Salad       3120
Sandwich    3074
Smoothie    3033
Tea         2954
Cookie      2898
UNKNOWN      883
ERROR        806
Name: quantity, dtype: Int64


In [89]:
if "item" in df.columns:
    item_revenue = (
        df.groupby("item")["revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    print(item_revenue)
# Revenue by Item

item
Salad       15600.0
Sandwich    12296.0
Smoothie    12132.0
Juice        9561.0
Cake         9540.0
Coffee       6424.0
Tea          4431.0
Cookie       2898.0
UNKNOWN      2550.0
ERROR        2395.5
Name: revenue, dtype: Float64


In [90]:
# Payment Method Analysis
if "payment_method" in df.columns:
    payment_analysis = (
        df.groupby("payment_method")["revenue"]
        .agg(["count", "sum", "mean"])
        .sort_values("sum", ascending=False)
    )

payment_analysis

,count,sum,mean
payment_method,,,
Digital Wallet,2068,18530.0,8.960348
Cash,2044,18486.0,9.044031
Credit Card,2047,18441.0,9.008793
ERROR,271,2380.5,8.784133
UNKNOWN,266,2326.5,8.746241


In [91]:
date_column = None

if "transaction_date" in df.columns:
    date_column = "transaction_date"

elif "date" in df.columns:
    date_column = "date"

if date_column:
    df["month"] = (
        df[date_column]
        .dt.to_period("M")
        .astype(str)
    )

    monthly_sales = (
        df.groupby("month")["revenue"]
        .sum()
        .sort_index()
    )

monthly_sales
 # Monthly Sales

month
2023-01    6452.5
2023-02    6055.0
2023-03    6482.0
2023-04    6485.5
2023-05    6191.0
2023-06    6678.0
2023-07    6366.5
2023-08    6453.0
2023-09    6209.5
2023-10    6512.0
2023-11    6418.0
2023-12    6387.5
NaT        3788.0
Name: revenue, dtype: Float64

In [92]:
highest_transaction = df.loc[
    df["revenue"].idxmax()
]

highest_transaction

transaction_id                TXN_2548360
item                                Salad
quantity                                5
price_per_unit                        5.0
total_spent                          25.0
payment_method                       Cash
location                         Takeaway
transaction_date      2023-11-07 00:00:00
calculated_revenue                   25.0
revenue_difference                    0.0
revenue                              25.0
month                             2023-11
Name: 10, dtype: object

In [93]:
print("Final Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Final Shape: (9006, 12)

Missing Values:
transaction_id           0
item                   293
quantity                 0
price_per_unit           0
total_spent            462
payment_method        2310
location              2941
transaction_date       410
calculated_revenue       0
revenue_difference     462
revenue                  0
month                    0
dtype: int64

Duplicate Rows:
0


In [94]:
output_folder = Path("../outputs")

output_folder.mkdir(exist_ok=True)

df.to_csv(
    output_folder / "cleaned_sales.csv",
    index=False
)

print("Cleaned CSV saved successfully!")

Cleaned CSV saved successfully!


In [95]:
with pd.ExcelWriter(
    output_folder / "sales_report.xlsx",
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        sheet_name="Cleaned Data",
        index=False
    )

    if "top_items" in locals():
        top_items.to_excel(
            writer,
            sheet_name="Top Items"
        )

    if "item_revenue" in locals():
        item_revenue.to_excel(
            writer,
            sheet_name="Item Revenue"
        )

    if "monthly_sales" in locals():
        monthly_sales.to_excel(
            writer,
            sheet_name="Monthly Sales"
        )

    if "payment_analysis" in locals():
        payment_analysis.to_excel(
            writer,
            sheet_name="Payment Analysis"
        )

print("Excel report saved successfully!")

Excel report saved successfully!


In [96]:
print("Cleaned CSV:")
print(output_folder / "cleaned_sales.csv")

print("\nExcel Report:")
print(output_folder / "sales_report.xlsx")

Cleaned CSV:
../outputs/cleaned_sales.csv

Excel Report:
../outputs/sales_report.xlsx
